# Ollama Local LLM Fallback

Audience: developers who want Knoema agents to keep running when cloud LLM providers are unavailable or too expensive for a demo.

Prerequisites: a local Python install of Knoema, optional Ollama, and one local instruct model such as Llama 3.3, Gemma 3, or Qwen 2.5.

By the end you can wire `OllamaClient` into `LLMGateway` with a deterministic fallback and inspect which provider answered.


## Outline

1. Prepare a small prompt.
2. Create an Ollama client and deterministic fallback.
3. Run the gateway in offline-safe mode.
4. Switch on live Ollama when the local server is available.
5. Try one extension exercise.


## Optional Ollama Setup

Run these commands outside the notebook if you want the live local path:

```powershell
ollama pull llama3.3
ollama serve
$env:KNOEMA_RUN_OLLAMA_DEMO='1'
$env:KNOEMA_OLLAMA_MODEL='llama3.3'
```

For lighter hardware, use `gemma3` or `qwen2.5` instead of `llama3.3`.


In [ ]:
from __future__ import annotations

import os

from knoema.llm import LLMGateway, LocalClient, OllamaClient

messages = [
    {"role": "system", "content": "You are a concise NPC simulation assistant."},
    {"role": "user", "content": "Give one safe next action for a student NPC studying late."},
]

messages


## Offline-Safe Gateway

The notebook defaults to a deterministic response so CI and readers without Ollama can still run it. Set `KNOEMA_RUN_OLLAMA_DEMO=1` to use the live local model first.


In [ ]:
run_live_ollama = os.getenv("KNOEMA_RUN_OLLAMA_DEMO") == "1"
ollama_model = os.getenv("KNOEMA_OLLAMA_MODEL", "llama3.3")

fallback = LocalClient(lambda _: "The student writes a short study plan, drinks water, and messages a roommate before sleeping.")
providers = []
if run_live_ollama:
    providers.append(("ollama", OllamaClient(model=ollama_model)))
providers.append(("deterministic-local", fallback))

gateway = LLMGateway(providers)
response = gateway.complete(messages, temperature=0.2, max_tokens=128)

response


## Inspect Provider Records

`LLMGateway.records` shows whether the local server answered or the deterministic fallback took over.


In [ ]:
[(record.provider, record.success, round(record.elapsed_seconds, 4), record.error) for record in gateway.records]


## Exercise

Change the prompt to a small game NPC interaction. If you have Ollama running, compare `llama3.3`, `gemma3`, and `qwen2.5` by changing `KNOEMA_OLLAMA_MODEL`.


In [ ]:
exercise_messages = [
    {"role": "system", "content": "You write short, non-violent NPC dialogue."},
    {"role": "user", "content": "A shopkeeper remembers the player bought tea yesterday. Reply in one sentence."},
]

# Answer scaffold: reuse the same gateway, then inspect gateway.records again.
exercise_response = gateway.complete(exercise_messages, temperature=0.4, max_tokens=80)
exercise_response


## Pitfall

If the Ollama server is not running and `KNOEMA_RUN_OLLAMA_DEMO=1`, the first provider will fail and the gateway will use the deterministic fallback. This is expected; check `gateway.records` before assuming the local model answered.
